# Notebook 10: Custom Data Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/10_custom_data_training.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Know how to capture images for 3DGS
2. Understand how to run COLMAP for pose estimation
3. Learn how to prepare custom datasets
4. Train 3DGS on your own data
5. Evaluate reconstruction quality

**Estimated Time**: 90 minutes

**Prerequisites**: All previous notebooks (00-09)

---

## Setup

In [ ]:
import os
import sys
import subprocess
import shutil

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Check for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Check for COLMAP
COLMAP_AVAILABLE = shutil.which('colmap') is not None
print(f"COLMAP available: {COLMAP_AVAILABLE}")

print("Setup complete!")

## 1. Image Capture Guidelines

### Best Practices for Capturing Images

| Aspect | Recommendation |
|--------|----------------|
| **Coverage** | Capture object from all angles (360°) |
| **Overlap** | 70-80% overlap between adjacent images |
| **Quantity** | 50-200 images for small objects |
| **Motion blur** | Avoid - use tripod or fast shutter |
| **Lighting** | Consistent, diffuse lighting |
| **Background** | Textured (not plain white) |
| **Focus** | Sharp focus on subject |

### Common Mistakes to Avoid

1. **Insufficient overlap** → COLMAP fails to match features
2. **Motion blur** → Poor feature extraction
3. **Changing lighting** → Inconsistent appearance
4. **Plain backgrounds** → COLMAP cannot find features
5. **Too few images** → Incomplete reconstruction

In [ ]:
# Visualize capture patterns
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Pattern 1: Orbital capture
ax = axes[0]
theta = np.linspace(0, 2*np.pi, 24, endpoint=False)
radius = 2
x = radius * np.cos(theta)
y = radius * np.sin(theta)

ax.scatter(x, y, s=100, c='blue', marker='^', label='Camera positions')
ax.scatter([0], [0], s=200, c='red', marker='o', label='Object')

# Draw viewing directions
for i in range(len(theta)):
    ax.arrow(x[i], y[i], -0.4*np.cos(theta[i]), -0.4*np.sin(theta[i]),
            head_width=0.1, head_length=0.05, fc='blue', ec='blue', alpha=0.5)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_title('Pattern 1: Orbital\n(360° around object)')
ax.legend()

# Pattern 2: Dome capture
ax = axes[1]
ax_3d = fig.add_subplot(1, 3, 2, projection='3d')

# Create dome points
phi = np.linspace(0, 2*np.pi, 12, endpoint=False)
theta_dome = np.linspace(0.3, np.pi/2, 4)

for t in theta_dome:
    x_dome = radius * np.sin(t) * np.cos(phi)
    y_dome = radius * np.sin(t) * np.sin(phi)
    z_dome = radius * np.cos(t) * np.ones_like(phi)
    ax_3d.scatter(x_dome, y_dome, z_dome, s=50, c='blue', marker='^')

ax_3d.scatter([0], [0], [0], s=200, c='red', marker='o')
ax_3d.set_title('Pattern 2: Dome\n(Multiple elevation angles)')

# Remove old 2D axis
axes[1].remove()

# Pattern 3: Walk-around (indoor scene)
ax = axes[2]

# Draw room
room = mpatches.Rectangle((-3, -3), 6, 6, fill=False, edgecolor='gray', linewidth=2)
ax.add_patch(room)

# Draw walking path
path_x = [-2, -2, -1, 0, 1, 2, 2, 2, 1, 0, -1, -2]
path_y = [-2, 0, 1, 1.5, 1, 0, -1, -2, -2.5, -2, -2.5, -2]
ax.plot(path_x, path_y, 'b--', linewidth=2, alpha=0.5)
ax.scatter(path_x, path_y, s=100, c='blue', marker='^', label='Camera positions')

# Draw objects
ax.scatter([0], [0], s=300, c='brown', marker='s', label='Furniture')
ax.scatter([-1], [1], s=150, c='green', marker='s')
ax.scatter([1.5], [-1], s=150, c='orange', marker='s')

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_title('Pattern 3: Walk-around\n(Indoor scene)')
ax.legend()

plt.tight_layout()
plt.show()

print("\nCapture Pattern Selection Guide:")
print("  - Small objects: Orbital or Dome pattern")
print("  - Indoor scenes: Walk-around pattern")
print("  - Outdoor scenes: Multiple trajectories with loops")

## 2. Dataset Directory Structure

### Expected Format for 3DGS

```
your_dataset/
├── input/                  # Your captured images
│   ├── image_001.jpg
│   ├── image_002.jpg
│   └── ...
├── sparse/                 # COLMAP output (created by convert.py)
│   └── 0/
│       ├── cameras.bin
│       ├── images.bin
│       └── points3D.bin
└── images/                 # Undistorted images (created by convert.py)
    ├── image_001.jpg
    ├── image_002.jpg
    └── ...
```

### Alternative: Blender Synthetic Format

```
blender_dataset/
├── transforms_train.json   # Camera poses for training
├── transforms_test.json    # Camera poses for testing
├── train/
│   ├── r_0.png
│   └── ...
└── test/
    ├── r_0.png
    └── ...
```

In [ ]:
# Create example dataset structure
def create_example_dataset_structure(base_path: str = "./example_dataset"):
    """
    Create example dataset directory structure.
    """
    base = Path(base_path)
    
    # Create directories
    dirs = [
        base / "input",
        base / "sparse" / "0",
        base / "images",
    ]
    
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)
        print(f"Created: {d}")
    
    # Create placeholder files
    (base / "input" / "README.txt").write_text(
        "Place your captured images here (*.jpg, *.png)\n"
        "Recommended: 50-200 images with 70-80% overlap\n"
    )
    
    (base / "sparse" / "0" / "README.txt").write_text(
        "COLMAP output files will be placed here:\n"
        "  - cameras.bin: Camera intrinsics\n"
        "  - images.bin: Camera poses (extrinsics)\n"
        "  - points3D.bin: Sparse 3D points\n"
    )
    
    return base


# Example usage
example_path = create_example_dataset_structure("./example_dataset")
print(f"\nExample dataset created at: {example_path.absolute()}")

## 3. COLMAP Processing Pipeline

### What is COLMAP?

COLMAP is a Structure-from-Motion (SfM) and Multi-View Stereo (MVS) pipeline that:
1. Extracts features from images (SIFT)
2. Matches features across images
3. Estimates camera poses (intrinsics + extrinsics)
4. Reconstructs sparse 3D point cloud

### Installation

```bash
# Ubuntu
sudo apt-get install colmap

# macOS
brew install colmap

# Or download from: https://colmap.github.io/install.html
```

In [ ]:
# COLMAP commands for 3DGS preprocessing
colmap_commands = '''
# Step 1: Feature Extraction
colmap feature_extractor \\
    --database_path $PROJECT/database.db \\
    --image_path $PROJECT/input \\
    --ImageReader.single_camera 1 \\
    --ImageReader.camera_model OPENCV \\
    --SiftExtraction.use_gpu 1

# Step 2: Feature Matching (exhaustive for small datasets)
colmap exhaustive_matcher \\
    --database_path $PROJECT/database.db \\
    --SiftMatching.use_gpu 1

# Step 3: Sparse Reconstruction (SfM)
mkdir -p $PROJECT/sparse
colmap mapper \\
    --database_path $PROJECT/database.db \\
    --image_path $PROJECT/input \\
    --output_path $PROJECT/sparse

# Step 4: Image Undistortion (optional but recommended)
mkdir -p $PROJECT/images
colmap image_undistorter \\
    --image_path $PROJECT/input \\
    --input_path $PROJECT/sparse/0 \\
    --output_path $PROJECT \\
    --output_type COLMAP
'''

print("COLMAP Processing Pipeline:")
print("=" * 60)
print(colmap_commands)

In [ ]:
def run_colmap_pipeline(project_path: str, use_gpu: bool = True):
    """
    Run complete COLMAP pipeline for 3DGS preprocessing.
    
    Args:
        project_path: Path to dataset with 'input' folder
        use_gpu: Whether to use GPU acceleration
    """
    project = Path(project_path)
    input_path = project / "input"
    db_path = project / "database.db"
    sparse_path = project / "sparse"
    
    if not input_path.exists():
        raise FileNotFoundError(f"Input folder not found: {input_path}")
    
    # Check for images
    images = list(input_path.glob("*.jpg")) + list(input_path.glob("*.png"))
    if len(images) == 0:
        raise ValueError(f"No images found in {input_path}")
    
    print(f"Found {len(images)} images in {input_path}")
    
    gpu_flag = "1" if use_gpu else "0"
    
    # Step 1: Feature extraction
    print("\nStep 1: Feature Extraction...")
    cmd_extract = [
        "colmap", "feature_extractor",
        "--database_path", str(db_path),
        "--image_path", str(input_path),
        "--ImageReader.single_camera", "1",
        "--ImageReader.camera_model", "OPENCV",
        "--SiftExtraction.use_gpu", gpu_flag,
    ]
    
    # Step 2: Feature matching
    print("\nStep 2: Feature Matching...")
    cmd_match = [
        "colmap", "exhaustive_matcher",
        "--database_path", str(db_path),
        "--SiftMatching.use_gpu", gpu_flag,
    ]
    
    # Step 3: Mapping (sparse reconstruction)
    print("\nStep 3: Sparse Reconstruction...")
    sparse_path.mkdir(parents=True, exist_ok=True)
    cmd_mapper = [
        "colmap", "mapper",
        "--database_path", str(db_path),
        "--image_path", str(input_path),
        "--output_path", str(sparse_path),
    ]
    
    # Step 4: Undistortion
    print("\nStep 4: Image Undistortion...")
    cmd_undistort = [
        "colmap", "image_undistorter",
        "--image_path", str(input_path),
        "--input_path", str(sparse_path / "0"),
        "--output_path", str(project),
        "--output_type", "COLMAP",
    ]
    
    commands = [
        ("Feature Extraction", cmd_extract),
        ("Feature Matching", cmd_match),
        ("Sparse Reconstruction", cmd_mapper),
        ("Image Undistortion", cmd_undistort),
    ]
    
    return commands


# Show what commands would be run
print("COLMAP Pipeline Commands:")
print("=" * 60)
try:
    commands = run_colmap_pipeline("./example_dataset")
    for name, cmd in commands:
        print(f"\n{name}:")
        print(f"  {' '.join(cmd)}")
except FileNotFoundError:
    print("(Example dataset not found - this is expected)")

## 4. Using Official 3DGS convert.py

The official 3DGS repository includes a `convert.py` script that automates COLMAP processing.

In [ ]:
# Official convert.py usage
convert_usage = '''
# Clone official repository
git clone https://github.com/graphdeco-inria/gaussian-splatting.git
cd gaussian-splatting

# Run convert.py on your dataset
python convert.py -s /path/to/your/dataset

# Options:
#   -s, --source_path     Path to dataset with 'input' folder
#   --skip_matching       Skip feature matching (if already done)
#   --colmap_executable   Path to COLMAP binary
#   --no_gpu              Disable GPU for COLMAP
#   --camera              Camera model (OPENCV, PINHOLE, etc.)
'''

print("Using Official convert.py:")
print("=" * 60)
print(convert_usage)

## 5. Training on Custom Data

### Training Command

```bash
python train.py -s /path/to/dataset
```

### Important Parameters

In [ ]:
# Training parameters
training_params = {
    'Essential': {
        '-s, --source_path': 'Path to dataset (required)',
        '-m, --model_path': 'Output path for trained model',
        '--iterations': 'Training iterations (default: 30000)',
    },
    'Optimization': {
        '--position_lr_init': 'Initial position LR (default: 0.00016)',
        '--position_lr_final': 'Final position LR (default: 0.0000016)',
        '--feature_lr': 'Feature LR (default: 0.0025)',
        '--opacity_lr': 'Opacity LR (default: 0.05)',
        '--scaling_lr': 'Scaling LR (default: 0.005)',
        '--rotation_lr': 'Rotation LR (default: 0.001)',
    },
    'Densification': {
        '--densify_from_iter': 'Start densification (default: 500)',
        '--densify_until_iter': 'End densification (default: 15000)',
        '--densification_interval': 'Densify every N iters (default: 100)',
        '--densify_grad_threshold': 'Gradient threshold (default: 0.0002)',
    },
    'Output': {
        '--save_iterations': 'Save at iterations (default: 7000 30000)',
        '--test_iterations': 'Test at iterations (default: 7000 30000)',
    },
}

print("Training Parameters:")
print("=" * 70)
for category, params in training_params.items():
    print(f"\n{category}:")
    for param, desc in params.items():
        print(f"  {param:30s} {desc}")

In [ ]:
# Example training commands
training_examples = '''
# Basic training
python train.py -s /path/to/dataset

# Training with custom output path
python train.py -s /path/to/dataset -m /path/to/output

# Training with fewer iterations (faster, lower quality)
python train.py -s /path/to/dataset --iterations 7000

# Training with more iterations (slower, higher quality)
python train.py -s /path/to/dataset --iterations 50000

# Training with custom save points
python train.py -s /path/to/dataset --save_iterations 1000 5000 10000 30000

# Training with white background (for objects)
python train.py -s /path/to/dataset --white_background

# Training with evaluation split
python train.py -s /path/to/dataset --eval
'''

print("Example Training Commands:")
print("=" * 60)
print(training_examples)

## 6. Rendering Trained Model

After training, render novel views:

In [ ]:
# Rendering commands
render_commands = '''
# Render training views
python render.py -m /path/to/output

# Render with specific iteration
python render.py -m /path/to/output --iteration 7000

# Skip training views, render test only
python render.py -m /path/to/output --skip_train

# Render with white background
python render.py -m /path/to/output --white_background
'''

print("Rendering Commands:")
print("=" * 60)
print(render_commands)

print("\nOutput structure after rendering:")
print("""
output/
├── point_cloud/
│   ├── iteration_7000/
│   │   └── point_cloud.ply
│   └── iteration_30000/
│       └── point_cloud.ply
├── train/
│   └── ours_30000/
│       ├── renders/        # Rendered images
│       └── gt/             # Ground truth images
└── test/
    └── ours_30000/
        ├── renders/
        └── gt/
""")

## 7. Evaluation Metrics

### Metrics Used in 3DGS

| Metric | Description | Range | Better |
|--------|-------------|-------|--------|
| **PSNR** | Peak Signal-to-Noise Ratio | 0-∞ | Higher |
| **SSIM** | Structural Similarity | 0-1 | Higher |
| **LPIPS** | Learned Perceptual Similarity | 0-1 | Lower |

In [ ]:
# Evaluation metrics implementation
import torch.nn.functional as F

def psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    """
    Calculate PSNR between predicted and target images.
    
    Args:
        pred: Predicted image [H, W, 3] or [3, H, W]
        target: Target image [H, W, 3] or [3, H, W]
    
    Returns:
        PSNR value in dB
    """
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(torch.tensor(1.0)) - 10 * torch.log10(mse)


def ssim(
    pred: torch.Tensor,
    target: torch.Tensor,
    window_size: int = 11,
) -> float:
    """
    Calculate SSIM between predicted and target images.
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    
    # Ensure NCHW format
    if pred.dim() == 3:
        if pred.shape[-1] == 3:
            pred = pred.permute(2, 0, 1)
            target = target.permute(2, 0, 1)
        pred = pred.unsqueeze(0)
        target = target.unsqueeze(0)
    
    # Create Gaussian kernel
    sigma = 1.5
    coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    g = torch.exp(-coords**2 / (2 * sigma**2))
    kernel_1d = g / g.sum()
    kernel_2d = kernel_1d.unsqueeze(1) @ kernel_1d.unsqueeze(0)
    kernel = kernel_2d.unsqueeze(0).unsqueeze(0)
    kernel = kernel.expand(3, 1, -1, -1).to(pred.device)
    
    pad = window_size // 2
    
    # Compute means
    mu1 = F.conv2d(F.pad(pred, (pad,pad,pad,pad), mode='reflect'),
                   kernel, groups=3)
    mu2 = F.conv2d(F.pad(target, (pad,pad,pad,pad), mode='reflect'),
                   kernel, groups=3)
    
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2
    
    # Compute variances
    sigma1_sq = F.conv2d(F.pad(pred**2, (pad,pad,pad,pad), mode='reflect'),
                         kernel, groups=3) - mu1_sq
    sigma2_sq = F.conv2d(F.pad(target**2, (pad,pad,pad,pad), mode='reflect'),
                         kernel, groups=3) - mu2_sq
    sigma12 = F.conv2d(F.pad(pred*target, (pad,pad,pad,pad), mode='reflect'),
                       kernel, groups=3) - mu1_mu2
    
    ssim_map = ((2*mu1_mu2 + C1) * (2*sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim_map.mean().item()


# Test metrics
print("Testing Evaluation Metrics:")
print("=" * 50)

# Create test images
target = torch.rand(256, 256, 3)
pred_good = target + torch.randn_like(target) * 0.05
pred_bad = target + torch.randn_like(target) * 0.2

pred_good = torch.clamp(pred_good, 0, 1)
pred_bad = torch.clamp(pred_bad, 0, 1)

print(f"\nGood prediction (low noise):")
print(f"  PSNR: {psnr(pred_good, target):.2f} dB")
print(f"  SSIM: {ssim(pred_good, target):.4f}")

print(f"\nBad prediction (high noise):")
print(f"  PSNR: {psnr(pred_bad, target):.2f} dB")
print(f"  SSIM: {ssim(pred_bad, target):.4f}")

In [ ]:
# Official metrics.py usage
metrics_usage = '''
# Calculate metrics on rendered images
python metrics.py -m /path/to/output

# Output: per-scene metrics (PSNR, SSIM, LPIPS)
'''

print("Metrics Evaluation:")
print("=" * 60)
print(metrics_usage)

print("\nTypical results on common datasets:")
print("""
┌────────────────┬──────────┬──────────┬──────────┐
│ Dataset        │ PSNR (dB)│ SSIM     │ LPIPS    │
├────────────────┼──────────┼──────────┼──────────┤
│ Mip-NeRF360    │ 27-29    │ 0.80-0.85│ 0.15-0.25│
│ Tanks&Temples  │ 23-25    │ 0.83-0.87│ 0.15-0.20│
│ Deep Blending  │ 29-30    │ 0.88-0.90│ 0.10-0.15│
│ Blender (NeRF) │ 32-34    │ 0.96-0.98│ 0.02-0.04│
└────────────────┴──────────┴──────────┴──────────┘
""")

## 8. Common Issues and Solutions

### Troubleshooting Guide

In [ ]:
troubleshooting = {
    "COLMAP fails to reconstruct": {
        "causes": [
            "Insufficient image overlap",
            "Motion blur in images",
            "Textureless surfaces",
            "Moving objects in scene",
        ],
        "solutions": [
            "Capture more images with 70-80% overlap",
            "Use tripod or faster shutter speed",
            "Add textured objects to scene",
            "Remove frames with moving objects",
        ],
    },
    "Training produces floaters": {
        "causes": [
            "Insufficient densification",
            "Background not well covered",
            "Sparse initial point cloud",
        ],
        "solutions": [
            "Increase densification iterations",
            "Add more background coverage",
            "Lower gradient threshold",
        ],
    },
    "Blurry results": {
        "causes": [
            "Too few Gaussians",
            "Insufficient training",
            "Large Gaussians not split",
        ],
        "solutions": [
            "Train for more iterations",
            "Lower split threshold",
            "Check densification is running",
        ],
    },
    "Out of GPU memory": {
        "causes": [
            "Too many Gaussians",
            "High resolution images",
            "Insufficient GPU VRAM",
        ],
        "solutions": [
            "Reduce image resolution",
            "Increase pruning threshold",
            "Use smaller batch of views",
        ],
    },
}

print("Troubleshooting Guide:")
print("=" * 70)

for issue, details in troubleshooting.items():
    print(f"\n{issue}")
    print("-" * len(issue))
    print("Causes:")
    for cause in details["causes"]:
        print(f"  - {cause}")
    print("Solutions:")
    for solution in details["solutions"]:
        print(f"  - {solution}")

## 9. Complete Workflow Example

In [ ]:
# Complete workflow script
workflow_script = '''
#!/bin/bash

# Complete 3DGS Training Workflow
# Usage: ./train_3dgs.sh /path/to/images output_name

IMAGE_DIR=$1
OUTPUT_NAME=$2

# Clone official repo if not exists
if [ ! -d "gaussian-splatting" ]; then
    git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
    cd gaussian-splatting
    pip install submodules/diff-gaussian-rasterization
    pip install submodules/simple-knn
    cd ..
fi

# Create dataset structure
DATASET_DIR="./datasets/${OUTPUT_NAME}"
mkdir -p ${DATASET_DIR}/input
cp ${IMAGE_DIR}/* ${DATASET_DIR}/input/

# Run COLMAP
echo "Running COLMAP..."
cd gaussian-splatting
python convert.py -s ${DATASET_DIR}

# Train
echo "Training 3DGS..."
python train.py -s ${DATASET_DIR} -m ./output/${OUTPUT_NAME}

# Render
echo "Rendering..."
python render.py -m ./output/${OUTPUT_NAME}

# Evaluate
echo "Evaluating..."
python metrics.py -m ./output/${OUTPUT_NAME}

echo "Done! Output saved to ./output/${OUTPUT_NAME}"
'''

print("Complete Workflow Script:")
print("=" * 60)
print(workflow_script)

## 10. Summary

### Custom Data Training Workflow

```
1. Capture Images
   └── 50-200 images, 70-80% overlap
   
2. Run COLMAP
   └── python convert.py -s /path/to/dataset
   
3. Train 3DGS
   └── python train.py -s /path/to/dataset
   
4. Render & Evaluate
   ├── python render.py -m /path/to/output
   └── python metrics.py -m /path/to/output
```

### Key Points

1. **Image quality matters** - Good captures = good reconstruction
2. **COLMAP is essential** - Provides camera poses and initialization
3. **Default parameters work well** - But can be tuned for specific scenes
4. **Evaluation is important** - Use PSNR, SSIM, LPIPS

---

## Congratulations!

You've completed **Phase 1** of the 3DGS tutorial series!

### What You've Learned

1. Gaussian distributions and 3D representation
2. Projection and splatting
3. Differentiable rendering
4. Alpha blending
5. Spherical harmonics
6. Adaptive density control
7. Training pipeline
8. Official code structure
9. Custom data training

### Next: Phase 2 - 3DGS + SLAM

In Phase 2, we'll explore:
- SLAM basics and integration
- SplaTAM architecture
- MonoGS implementation
- Online Gaussian optimization